# Roster and Stat Scraper for Elite Prospects Site
- because CHN's roster data is not always accurate I want to get a second source to check it against.
- after some very simple exploration on hockeydb.com I was flagged and IP banned for scraping, though there site TOS says they allow limited scraping of their data for non commercial use.
    - the commented out code is some that was developed for hockeydb.com but not tested bvery much
- after being IP banned (it was removed a few hours later) I decided to look for another source and found elit prospects which seems more open to the type of scraping I am trying

In [1]:
# Updated cleaner with multi-position handling: Pos_OVR, Pos_Prime, Pos_Second
import pandas as pd
import numpy as np
import re

elite_path = "../TEMP//elite_prospects_current_team_roster_temp.csv"
elite = pd.read_csv(elite_path)

ALLOWED_POS = {"G","D","F","LW","C","RW"}

def parse_player(player_str: str):
    """Extract (name, position_text, captaincy) from the Elite Prospects Player field."""
    if not isinstance(player_str, str):
        return None, None, None
    cap = re.search(r'\"([AC])\"', player_str)
    captaincy = cap.group(1) if cap else None
    cleaned = re.sub(r'\s*\"[AC]\"\s*', '', player_str)
    posm = re.search(r'\(([^)]+)\)\s*$', cleaned)
    position_text = posm.group(1).strip().upper() if posm else None
    name = re.sub(r'\s*\([^)]+\)\s*$', '', cleaned).strip()
    return name, position_text, captaincy

def split_name(full_name: str):
    if not isinstance(full_name, str) or not full_name.strip():
        return None, None
    parts = full_name.strip().split()
    if len(parts) == 1:
        return parts[0], None
    return " ".join(parts[:-1]), parts[-1]

def ht_to_inches(ht_str: str):
    if not isinstance(ht_str, str):
        return None
    m = re.match(r"^\s*(\d+)'\s*(\d+)\"\s*$", ht_str)
    if not m:
        return None
    return int(m.group(1))*12 + int(m.group(2))

def split_birthplace(bp: str):
    if not isinstance(bp, str):
        return None, None, None
    parts = [p.strip() for p in str(bp).split(",")]
    if len(parts) == 3:
        city, sp, country = parts
    elif len(parts) == 2:
        city, country = parts; sp = None
    else:
        city, sp, country = None, None, parts[0] if parts else None
    return city, sp, country

def normalize_positions(position_text: str):
    """
    Returns Pos_OVR, Pos_Prime, Pos_Second:
      - Pos_OVR: {G,D,F} (C/LW/RW/F -> F)
      - Pos_Prime: first of [G,D,F,LW,C,RW]
      - Pos_Second: second token if present; else None
    """
    if not isinstance(position_text, str) or not position_text.strip():
        return None, None, None
    tokens = [t.strip().upper() for t in position_text.split("/") if t.strip()]
    tokens = [t for t in tokens if t in ALLOWED_POS]
    if not tokens:
        return None, None, None

    pos_prime = tokens[0]
    pos_second = tokens[1] if len(tokens) > 1 else None

    if pos_prime == "G":
        pos_ovr = "G"
    elif pos_prime == "D":
        pos_ovr = "D"
    else:
        pos_ovr = "F"
    return pos_ovr, pos_prime, pos_second

def clean_elite_prospects(df: pd.DataFrame) -> pd.DataFrame:
    # 1) drop junk + section headers
    df = df.drop(columns=[c for c in ['Unnamed: 0','N'] if c in df.columns], errors='ignore').copy()
    headers = {'GOALTENDERS','DEFENSEMEN','FORWARDS'}
    df = df[~df['Player'].isin(headers)].copy()

    # 2) parse Player → (name, positions, captaincy)
    df[['Full_Name','Position_text','Captaincy']] = pd.DataFrame(
        df['Player'].apply(parse_player).tolist(), index=df.index
    )

    # 3) split names
    df[['First_Name','Last_Name']] = pd.DataFrame(
        df['Full_Name'].apply(split_name).tolist(), index=df.index
    )

    # 4) jersey No
    df['No'] = (
        df.get('#', pd.Series(index=df.index, dtype='object')).astype(str)
          .str.replace('#','', regex=False).str.strip()
          .replace({'nan': None, 'None': None, '': None})
    )

    # 5) height/weight/dob
    df['Height_Inches'] = df['HT'].apply(ht_to_inches)
    df['DOB'] = df['Born'].astype(str).replace({'nan': None})
    df['Wt'] = pd.to_numeric(df['WT'], errors='coerce')
    df['Ht'] = df['HT']
    df['Shoots'] = df['S'] if 'S' in df.columns else None

    # 6) birthplace
    df[['City','State_Province','Country']] = pd.DataFrame(
        df['Birthplace'].apply(split_birthplace).tolist(), index=df.index
    )
    df['Hometown'] = df['Birthplace']

    # 7) multi-position mapping
    df[['Pos_OVR','Pos_Prime','Pos_Second']] = pd.DataFrame(
        df['Position_text'].apply(normalize_positions).tolist(), index=df.index
    )
    # keep Position for compatibility with your existing tools
    df['Position'] = df['Pos_Prime']

    # 8) placeholders to match your schema
    for c in ['Yr','Draft_Year','NHL_Team','D_Round','Last Team','League','Current Team']:
        if c not in df.columns:
            df[c] = None

    # 9) final column order
    target_cols = [
        'Current Team','Last_Name','First_Name','No',
        'Position','Pos_OVR','Pos_Prime','Pos_Second',
        'Yr','Ht','Wt','DOB','Hometown','Height_Inches',
        'Draft_Year','NHL_Team','D_Round','Last Team','League',
        'City','State_Province','Country','Captaincy','Shoots'
    ]
    for c in target_cols:
        if c not in df.columns:
            df[c] = None
    return df[target_cols].reset_index(drop=True)

# Example usage on your uploaded sample:
cleaned = clean_elite_prospects(elite)


out_path = "../TEMP/DATA/elite_prospects_cleaned_with_positions.csv"
cleaned.to_csv(out_path, index=False)
out_path



'../TEMP/DATA/elite_prospects_cleaned_with_positions.csv'

In [2]:
## Display Cleaned Sample
cleaned.head(10)

,Current Team,Last_Name,First_Name,No,Position,Pos_OVR,Pos_Prime,Pos_Second,Yr,Ht,...,Draft_Year,NHL_Team,D_Round,Last Team,League,City,State_Province,Country,Captaincy,Shoots
0,None,Augustine,Trey,1,G,G,G,None,None,"6'1""",...,None,None,None,None,None,South Lyon,MI,USA,None,L
1,None,Gilbert,Dolan,30,G,G,G,None,None,"6'2""",...,None,None,None,None,None,South Bend,IN,USA,None,L
2,None,Strahl,Melvin,32,G,G,G,None,None,"6'3""",...,None,None,None,None,None,Sollefteå,None,SWE,None,L
3,None,Barnhill,Sean,3,D,D,D,None,None,"6'6""",...,None,None,None,None,None,Scottsdale,AZ,USA,None,R
4,None,Basgall,Matt,9,D,D,D,None,None,"5'10""",...,None,None,None,None,None,Lake Forest,IL,USA,C,R
5,None,Geary,Patrick,2,D,D,D,None,None,"6'1""",...,None,None,None,None,None,Hamburg,NY,USA,A,L
6,None,Lahey,Matthew,14,D,D,D,None,None,"6'6""",...,None,None,None,None,None,Victoria,BC,CAN,None,L
7,None,Ralph,Colin,4,D,D,D,None,None,"6'5""",...,None,None,None,None,None,Maple Grove,MN,USA,None,L
8,None,Shoudy,Travis,5,D,D,D,None,None,"5'10""",...,None,None,None,None,None,Marysville,MI,USA,A,L
9,None,Strbak,Maxim,8,D,D,D,None,None,"6'2""",...,None,None,None,None,None,Kosice,None,SVK,None,R


In [3]:
# Create Table of teams and roster URLs

# Load school info including links to EP.com
school_info_df = pd.read_csv('../data/school_info/arena_school_info.csv')

# Load and check
school_info_df.head()

,Team,Arena,Capacity,Sheet_length,Sheet_width,School,Latitude,Longitude,hex1,hex2,hex3,simp_color,logo_abv,abv,ncaa_name,ncaa_data_alts,eliteprospects_url
0,Air Force,Cadet Ice Arena,2470,200,85,Air Force,39.013739,-104.883727,3087,8a8d8f,NaN,NaN,afa,Air Force,Air Force,"AIRFOR, Air Force",https://www.eliteprospects.com/team/2453/air-f...
1,Alaska,Carlson Center,4595,200,100,Alaska,64.842124,-147.763841,236192,ffcd00,NaN,NaN,akf,Alaska,Alas Fairbanks,"AK FBK, Alas. Fairbanks",https://www.eliteprospects.com/team/2071/univ....
2,Alaska Anchorage,Avis Alaska Sports Complex,800,200,85,Alaska-Anchorage,61.205536,-149.872737,00583d,ffc425,NaN,NaN,aka,UAA,Alas Anchorage,"AK ANC, Alas. Anchorage",https://www.eliteprospects.com/team/1915/univ....
3,American Intl,MassMutual Center,6866,200,85,American Int'l,42.118003,-72.554326,0,ffb60f,NaN,NaN,aic,AIC,American Intl,"AM INT, American Int'l",https://www.eliteprospects.com/team/1252/ameri...
4,American Int'l,MassMutual Center,6866,200,85,American Int'l,42.118003,-72.554326,0,ffb60f,NaN,NaN,aic,AIC,American Intl,"AM INT, American Int'l",https://www.eliteprospects.com/team/1252/ameri...


In [4]:
# simplify to just team name and roster URL
school_info_df = school_info_df[['Team', 'eliteprospects_url']]
# Save to temp csv file
school_info_df.to_csv('../TEMP/school_info_temp.csv', index=False)

In [5]:
# Create an improved test block addressing: 
# - FutureWarning (wrap HTML in StringIO)
# - More robust table detection & column normalization
# - Better error surfacing (prints + logs)
# - Auto-ensure '?tab=roster' on EP URLs when missing
#
# The cell reads your 'school_info_temp.csv', samples 3 teams, and runs.

# JUPYTER TEST BLOCK v2 — EliteProspects Roster Scraper (robust parse, 3 random teams)
# Requirements: pandas, requests, beautifulsoup4, lxml
import os, time, random, logging, re
from pathlib import Path
from io import StringIO
import pandas as pd
import requests
from bs4 import BeautifulSoup


# -----------------------------
# Config & folders
# -----------------------------
BASE = Path("../TEMP/EP_SCRAPER/")
RAW_HTML_DIR = BASE / "raw_html"
RAW_CSV_DIR  = BASE / "raw_csv"
CLEAN_DIR    = BASE / "clean_csv"
LOG_DIR      = BASE / "logs"
OUT_DIR      = BASE / "out"
for d in [RAW_HTML_DIR, RAW_CSV_DIR, CLEAN_DIR, LOG_DIR, OUT_DIR]:
    d.mkdir(parents=True, exist_ok=True)
# # -----------------------------
# # Config & folders
# # -----------------------------
# BASE = Path(".")
# RAW_HTML_DIR = BASE / "raw_html"
# RAW_CSV_DIR  = BASE / "raw_csv"
# CLEAN_DIR    = BASE / "clean_csv"
# LOG_DIR      = BASE / "logs"
# OUT_DIR      = BASE / "out"
# for d in [RAW_HTML_DIR, RAW_CSV_DIR, CLEAN_DIR, LOG_DIR, OUT_DIR]:
#     d.mkdir(parents=True, exist_ok=True)

logger = logging.getLogger("ep_rosters")
logger.setLevel(logging.INFO)
# Log to file + console
fh = logging.FileHandler(LOG_DIR / "ep_roster_scrape.log")
fh.setLevel(logging.INFO)
ch = logging.StreamHandler()
ch.setLevel(logging.INFO)
fmt = logging.Formatter("%(asctime)s %(levelname)s %(message)s")
fh.setFormatter(fmt); ch.setFormatter(fmt)
# Avoid duplicate handlers in repeated runs
if not logger.handlers:
    logger.addHandler(fh); logger.addHandler(ch)

HEADERS = {
    "User-Agent": "NCAADataSauce Roster Builder (research use; contact: youremail@example.com)",
    "From": "youremail@example.com",
    "Accept-Language": "en-US,en;q=0.9",
}

# -----------------------------
# Cleaner (with Pos_OVR / Pos_Prime / Pos_Second)
# -----------------------------
ALLOWED_POS = {"G","D","F","LW","C","RW"}

def parse_player(player_str: str):
    if not isinstance(player_str, str):
        return None, None, None
    cap = re.search(r'\"([AC])\"', player_str)
    captaincy = cap.group(1) if cap else None
    cleaned = re.sub(r'\s*\"[AC]\"\s*', '', player_str)
    posm = re.search(r'\(([^)]+)\)\s*$', cleaned)
    position_text = posm.group(1).strip().upper() if posm else None
    name = re.sub(r'\s*\([^)]+\)\s*$', '', cleaned).strip()
    return name, position_text, captaincy

def split_name(full_name: str):
    if not isinstance(full_name, str) or not full_name.strip():
        return None, None
    parts = full_name.strip().split()
    if len(parts) == 1:
        return parts[0], None
    return " ".join(parts[:-1]), parts[-1]

def ht_to_inches(ht_str: str):
    if not isinstance(ht_str, str):
        return None
    m = re.match(r"^\s*(\d+)'\s*(\d+)\"\s*$", ht_str)
    if not m:
        return None
    return int(m.group(1))*12 + int(m.group(2))

def split_birthplace(bp: str):
    if not isinstance(bp, str):
        return None, None, None
    parts = [p.strip() for p in str(bp).split(",")]
    if len(parts) == 3:
        city, sp, country = parts
    elif len(parts) == 2:
        city, country = parts; sp = None
    else:
        city, sp, country = None, None, parts[0] if parts else None
    return city, sp, country

def normalize_positions(position_text: str):
    if not isinstance(position_text, str) or not position_text.strip():
        return None, None, None
    tokens = [t.strip().upper() for t in position_text.split("/") if t.strip()]
    tokens = [t for t in tokens if t in ALLOWED_POS]
    if not tokens:
        return None, None, None
    pos_prime = tokens[0]
    pos_second = tokens[1] if len(tokens) > 1 else None
    pos_ovr = "G" if pos_prime == "G" else ("D" if pos_prime == "D" else "F")
    return pos_ovr, pos_prime, pos_second

def clean_elite_prospects(df: pd.DataFrame) -> pd.DataFrame:
    df = df.drop(columns=[c for c in ['Unnamed: 0','N'] if c in df.columns], errors='ignore').copy()
    headers = {'GOALTENDERS','DEFENSEMEN','FORWARDS'}
    if 'Player' in df.columns:
        df = df[~df['Player'].isin(headers)].copy()

    df[['Full_Name','Position_text','Captaincy']] = pd.DataFrame(
        df['Player'].apply(parse_player).tolist(), index=df.index
    )
    df[['First_Name','Last_Name']] = pd.DataFrame(
        df['Full_Name'].apply(split_name).tolist(), index=df.index
    )
    df['No'] = (
        df.get('#', pd.Series(index=df.index, dtype='object')).astype(str)
          .str.replace('#','', regex=False).str.strip()
          .replace({'nan': None, 'None': None, '': None})
    )
    df['Height_Inches'] = df['HT'].apply(ht_to_inches) if 'HT' in df.columns else None
    df['DOB'] = df['Born'].astype(str).replace({'nan': None}) if 'Born' in df.columns else None
    df['Wt'] = pd.to_numeric(df['WT'], errors='coerce') if 'WT' in df.columns else None
    df['Ht'] = df['HT'] if 'HT' in df.columns else None
    df['Shoots'] = df['S'] if 'S' in df.columns else None

    if 'Birthplace' in df.columns:
        df[['City','State_Province','Country']] = pd.DataFrame(
            df['Birthplace'].apply(split_birthplace).tolist(), index=df.index
        )
        df['Hometown'] = df['Birthplace']
    else:
        for c in ['City','State_Province','Country','Hometown']:
            df[c] = None

    df[['Pos_OVR','Pos_Prime','Pos_Second']] = pd.DataFrame(
        df['Position_text'].apply(normalize_positions).tolist(), index=df.index
    )
    df['Position'] = df['Pos_Prime']

    for c in ['Yr','Draft_Year','NHL_Team','D_Round','Last Team','League','Current Team']:
        if c not in df.columns:
            df[c] = None

    target_cols = [
        'Current Team','Last_Name','First_Name','No',
        'Position','Pos_OVR','Pos_Prime','Pos_Second',
        'Yr','Ht','Wt','DOB','Hometown','Height_Inches',
        'Draft_Year','NHL_Team','D_Round','Last Team','League',
        'City','State_Province','Country','Captaincy','Shoots'
    ]
    for c in target_cols:
        if c not in df.columns:
            df[c] = None
    return df[target_cols].reset_index(drop=True)

# -----------------------------
# Robust parsing helpers
# -----------------------------
def flatten_cols(cols):
    out = []
    for c in cols:
        if isinstance(c, tuple):
            c = " ".join([str(x) for x in c if str(x).lower() != "nan"]).strip()
        out.append(str(c).strip())
    return out

def normalize_header_name(name: str) -> str:
    n = name.strip().lower()
    # Map common header variants to our expected ones
    if n in {"player","players","name"}: return "Player"
    if n in {"born","birth","birthdate","date of birth"}: return "Born"
    if n in {"ht","height"}: return "HT"
    if n in {"wt","weight"}: return "WT"
    if n in {"s","shoots","shot"}: return "S"
    if n in {"ctrct","contract"}: return "CTRCT"
    if n in {"#","no","no."}: return "#"
    if n in {"birth place","birthplace","place of birth"}: return "Birthplace"
    return name.strip()

def coerce_column_names(df: pd.DataFrame) -> pd.DataFrame:
    # Flatten any MultiIndex and normalize known synonyms
    df = df.copy()
    df.columns = flatten_cols(df.columns)
    df.columns = [normalize_header_name(c) for c in df.columns]
    return df

def looks_like_roster(df: pd.DataFrame) -> bool:
    cols = set([c.lower() for c in df.columns])
    return ("player" in cols) and (("born" in cols) or ("ht" in cols) or ("birthplace" in cols))

def parse_roster_table(html: str) -> pd.DataFrame:
    # Quick guard: detect bot/cookie walls
    low = html.lower()
    if any(s in low for s in ["enable javascript", "captcha", "cloudflare", "access denied"]):
        raise ValueError("Page appears to be a bot/cookie wall (no roster table in HTML).")

    # Preferred: pandas.read_html on a StringIO (avoids FutureWarning)
    tables = []
    try:
        tables = pd.read_html(StringIO(html))
    except ValueError:
        pass

    candidates = []
    for t in tables:
        t = coerce_column_names(t)
        if looks_like_roster(t) and len(t) >= 5:
            candidates.append(t)

    if candidates:
        # Choose the widest (most columns), tie-break by length
        candidates.sort(key=lambda d: (d.shape[1], d.shape[0]), reverse=True)
        return candidates[0]

    # Fallback: use BeautifulSoup to find the specific table that contains a "Player" header
    soup = BeautifulSoup(html, "html.parser")
    for tbl in soup.find_all("table"):
        df = pd.read_html(StringIO(str(tbl)))[0]
        df = coerce_column_names(df)
        if looks_like_roster(df) and len(df) >= 5:
            return df

    raise ValueError("No roster-like table found.")

# -----------------------------
# Fetch + Parse
# -----------------------------
def polite_sleep(min_s: float, max_s: float):
    time.sleep(random.uniform(min_s, max_s))

def fix_url(url: str) -> str:
    """Ensure we're loading the roster tab; if already has query params, append safely."""
    try:
        from urllib.parse import urlparse, parse_qs, urlencode, urlunparse
        u = urlparse(url)
        q = parse_qs(u.query)
        if 'tab' not in q:
            q['tab'] = ['roster']
        new_q = urlencode({k: v[0] if isinstance(v, list) else v for k, v in q.items()})
        return urlunparse((u.scheme, u.netloc, u.path, u.params, new_q, u.fragment))
    except Exception:
        return url

def fetch_html(url: str, session: requests.Session, refresh: bool, cache_path: Path):
    if cache_path.exists() and not refresh:
        return cache_path.read_text(encoding="utf-8", errors="ignore")
    r = session.get(url, headers=HEADERS, timeout=30)
    if r.status_code == 429:
        logger.warning(f"429 Too Many Requests for {url}.")
        raise RuntimeError("429")
    r.raise_for_status()
    html = r.text
    cache_path.write_text(html, encoding="utf-8")
    return html

# -----------------------------
# Team CSV helpers (your format: Team, eliteprospects_url)
# -----------------------------
def read_teams_csv(csv_path: str) -> pd.DataFrame:
    teams = pd.read_csv(csv_path)
    need = {'Team','eliteprospects_url'}
    if not need.issubset(set(teams.columns)):
        raise ValueError("CSV must contain columns: 'Team' and 'eliteprospects_url'")
    teams = teams.rename(columns={'eliteprospects_url': 'EP_Roster_URL'})
    teams['EP_Roster_URL'] = teams['EP_Roster_URL'].astype(str).apply(fix_url)
    return teams[['Team','EP_Roster_URL']]

def pick_random_subset(teams_df: pd.DataFrame, n: int = 3, seed: int | None = None) -> pd.DataFrame:
    return teams_df.sample(n=n, random_state=seed).reset_index(drop=True)

# -----------------------------
# Main builder (from a given teams DF)
# -----------------------------
def build_master_from_df(teams_df: pd.DataFrame, sleep_min: float = 8.0, sleep_max: float = 15.0, refresh: bool = False):
    all_clean = []
    session = requests.Session()

    for _, row in teams_df.iterrows():
        team = row["Team"]
        url = row["EP_Roster_URL"]
        safe_team = re.sub(r'[^A-Za-z0-9]+','_', team).strip("_")
        html_path = RAW_HTML_DIR / f"{safe_team}.html"
        raw_csv_path = RAW_CSV_DIR / f"{safe_team}.csv"
        clean_path   = CLEAN_DIR / f"{safe_team}_clean.csv"

        try:
            html = fetch_html(url, session, refresh, html_path)
            df_raw = parse_roster_table(html)
            df_raw.to_csv(raw_csv_path, index=False)

            if 'Player' not in df_raw.columns:
                raise ValueError(f"Parsed table missing 'Player' column. Columns={list(df_raw.columns)}")

            df_clean = clean_elite_prospects(df_raw)
            df_clean["Current Team"] = team
            df_clean.to_csv(clean_path, index=False)

            all_clean.append(df_clean)
            logger.info(f"OK: {team} ({len(df_clean)} rows)")
        except Exception as e:
            logger.exception(f"FAILED: {team} -> {e}")
        polite_sleep(sleep_min, sleep_max)

    if all_clean:
        master = pd.concat(all_clean, ignore_index=True)
        out_partial = OUT_DIR / "all_d1_master.partial.csv"
        master.to_csv(out_partial, index=False)
        return master
    else:
        return pd.DataFrame()

# # -----------------------------
# # TEST RUN: pick 3 random teams from your CSV and build
# # -----------------------------
# TEAM_CSV = "../TEMP/school_info_temp.csv"  # update path if needed
# teams_df = read_teams_csv(TEAM_CSV)
# subset_df = pick_random_subset(teams_df, n=3, seed=None)  # set seed for reproducibility if desired
# print("Selected teams:")
# print(subset_df)

# master_df = build_master_from_df(subset_df, sleep_min=8.0, sleep_max=15.0, refresh=False)
# print(f"Collected {len(master_df)} rows from {len(subset_df)} teams.")
# display(master_df.head(20))

# '''
# path = "/mnt/data/ep_roster_scrape_block_v2.py"
# with open(path, "w", encoding="utf-8") as f:
#     f.write(improved_code)

# path


#### Run the Elite Prospect Roster Scraper
- run for all teams in the school info 

In [6]:
# Run for all teams on NCAA D1 List and save to CSV
TEAM_CSV = "../data/school_info/arena_school_info.csv"  # update path if needed
teams_df = read_teams_csv(TEAM_CSV)
# subset_df = pick_random_subset(teams_df, n=3, seed=None)  # set seed for reproducibility if desired
print("Selected teams:")
# print(subset_df)
print(teams_df)

master_df = build_master_from_df(teams_df, sleep_min=5.0, sleep_max=10.0, refresh=False)
print(f"Collected {len(master_df)} rows from {len(subset_df)} teams.")
display(master_df.head(20))

Selected teams:
                Team                                      EP_Roster_URL
0          Air Force  https://www.eliteprospects.com/team/2453/air-f...
1             Alaska  https://www.eliteprospects.com/team/2071/univ....
2   Alaska Anchorage  https://www.eliteprospects.com/team/1915/univ....
3      American Intl  https://www.eliteprospects.com/team/1252/ameri...
4     American Int'l  https://www.eliteprospects.com/team/1252/ameri...
..               ...                                                ...
64             Union  https://www.eliteprospects.com/team/1366/union...
65           Vermont  https://www.eliteprospects.com/team/710/univ.-...
66  Western Michigan  https://www.eliteprospects.com/team/1250/weste...
67         Wisconsin  https://www.eliteprospects.com/team/452/univ.-...
68              Yale  https://www.eliteprospects.com/team/786/yale-u...

[69 rows x 2 columns]


2025-09-14 23:45:02,435 INFO OK: Air Force (32 rows)
2025-09-14 23:45:10,620 INFO OK: Alaska (31 rows)
2025-09-14 23:45:19,918 INFO OK: Alaska Anchorage (29 rows)
2025-09-14 23:45:26,651 INFO OK: American Intl (33 rows)
2025-09-14 23:45:34,585 INFO OK: American Int'l (33 rows)
2025-09-14 23:45:42,102 INFO OK: Arizona State (27 rows)
2025-09-14 23:45:51,970 INFO OK: Army (30 rows)
2025-09-14 23:46:00,567 INFO OK: Augustana (28 rows)
2025-09-14 23:46:09,826 INFO OK: Bemidji State (28 rows)
2025-09-14 23:46:19,227 INFO OK: Bentley (31 rows)
2025-09-14 23:46:27,800 INFO OK: Boston College (27 rows)
2025-09-14 23:46:35,449 INFO OK: Boston University (25 rows)
2025-09-14 23:46:44,735 INFO OK: Bowling Green (28 rows)
2025-09-14 23:46:54,343 INFO OK: Brown (28 rows)
2025-09-14 23:47:04,790 INFO OK: Canisius (30 rows)
2025-09-14 23:47:12,883 INFO OK: Clarkson (26 rows)
2025-09-14 23:47:25,236 INFO OK: Colgate (26 rows)
2025-09-14 23:47:37,049 INFO OK: Colorado College (27 rows)
2025-09-14 23:47

NameError: name 'subset_df' is not defined

##### Save The CSV of EP Data

In [7]:
## Save The master CSV 

output_path = '../data/player_info/EP_master_roster_v0.2_9-14.csv'
master_df.to_csv(output_path, index=False)

## New Testing Block
- Junior Stats from 2024-25 season from EP

In [ ]:
sample_url = 'https://www.eliteprospects.com/team/857/brandon-wheat-kings/2024-2025?tab=stats' # 2034-35 Stats

# sample_url = 'https://www.eliteprospects.com/team/1157/michigan-state-univ.' # Current Roster Test

### Read with pandas
import pandas as pd

# dfs = pd.read_html(sample_url)
# dfs[0].head(10)

# ## Print summary of every dataframe in list
# for i, df in enumerate(dfs):
#     print(f"DataFrame {i} summary:")
#     print(df.info())
#     print("\n") 

In [ ]:
# # Create a ready-to-run Jupyter cell that scrapes EP team stats (Skaters + Goalies)
# # from the React block the user pasted, using robust BeautifulSoup parsing
# # and normalizing columns so they can be joined to your NCAA roster table later.
# #
# # The cell includes:
# # - fix_url_stats(url, season) to enforce ?tab=stats and optional season
# # - fetch_html (polite headers + caching)
# # - parse_stats_tables(html) -> skaters_df_raw, goalies_df_raw
# # - clean_stats_skaters / clean_stats_goalies to produce normalized tables with:
# #     Team, Season, League, Player_Name, First_Name, Last_Name, Pos_OVR/Prime/Second,
# #     EP_Player_ID, EP_Player_URL, and stat columns (regular + playoffs)
# # - save CSVs to ./out/
# # - demo harness (commented) to run on one URL
# #
# # NOTE: This cell is designed for your local machine. It won't fetch live pages here.


# # JUPYTER BLOCK — EliteProspects Team Stats (Skaters + Goalies)
# # Requirements: pandas, requests, beautifulsoup4, lxml
# import os, re, time, random, logging
# from io import StringIO
# from pathlib import Path
# import pandas as pd
# import requests
# from bs4 import BeautifulSoup

# try:
#     from unidecode import unidecode
# except Exception:
#     unidecode = None  # optional; used for name keys

# # -----------------------------
# # Config & folders
# # -----------------------------
# BASE = Path("../TEMP/JR_STATS")
# RAW_HTML_DIR = BASE / "raw_html_stats"
# OUT_DIR      = BASE / "out"
# LOG_DIR      = BASE / "logs"
# for d in [RAW_HTML_DIR, OUT_DIR, LOG_DIR]:
#     d.mkdir(parents=True, exist_ok=True)

# logger = logging.getLogger("ep_stats")
# logger.setLevel(logging.INFO)
# if not logger.handlers:
#     fh = logging.FileHandler(LOG_DIR / "ep_stats_scrape.log")
#     fh.setFormatter(logging.Formatter("%(asctime)s %(levelname)s %(message)s"))
#     logger.addHandler(fh)
#     ch = logging.StreamHandler()
#     ch.setFormatter(logging.Formatter("%(levelname)s %(message)s"))
#     logger.addHandler(ch)

# HEADERS = {
#     "User-Agent": "NCAADataSauce Stats Builder (research use; contact: youremail@example.com)",
#     "From": "youremail@example.com",
#     "Accept-Language": "en-US,en;q=0.9",
# }

# ALLOWED_POS = {"G","D","F","LW","C","RW","W"}  # 'W' appears on EP; we map it

# def polite_sleep(min_s: float = 8.0, max_s: float = 15.0):
#     time.sleep(random.uniform(min_s, max_s))

# def fix_url_stats(url: str, season: str | None = None) -> str:
#     """Ensure we hit the stats tab and (optionally) a specific season, e.g., '2023-2024'."""
#     from urllib.parse import urlparse, parse_qs, urlencode, urlunparse
#     u = urlparse(url)
#     q = parse_qs(u.query)
#     q["tab"] = ["stats"]
#     if season:
#         q["season"] = [season]
#     new_q = urlencode({k: v[0] if isinstance(v, list) else v for k, v in q.items()})
#     return urlunparse((u.scheme, u.netloc, u.path, u.params, new_q, u.fragment))

# def fetch_html(url: str, cache_name: str, refresh: bool = False) -> str:
#     cache_path = RAW_HTML_DIR / f"{cache_name}.html"
#     if cache_path.exists() and not refresh:
#         return cache_path.read_text(encoding="utf-8", errors="ignore")
#     r = requests.get(url, headers=HEADERS, timeout=30)
#     if r.status_code == 429:
#         raise RuntimeError("429 Too Many Requests")
#     r.raise_for_status()
#     html = r.text
#     cache_path.write_text(html, encoding="utf-8")
#     return html

# # -----------------------------
# # Helpers
# # -----------------------------
# def text(el):
#     return el.get_text(strip=True) if el else ""

# def extract_player_link(td):
#     """Return (player_url, player_id, player_name_raw) from the player cell."""
#     a = td.find("a")
#     if not a:
#         return None, None, text(td)
#     href = a.get("href", "")
#     # EP relative → absolute
#     player_url = "https://www.eliteprospects.com" + href if href.startswith("/") else href
#     m = re.search(r"/player/(\d+)/", href)
#     player_id = m.group(1) if m else None
#     return player_url, player_id, text(td)

# def split_name(full_name: str):
#     if not isinstance(full_name, str) or not full_name.strip():
#         return None, None
#     parts = full_name.strip().split()
#     if len(parts) == 1:
#         return parts[0], None
#     return " ".join(parts[:-1]), parts[-1]

# def parse_player_field(player_raw: str):
#     """From 'Nolan Flamand (F)' → ('Nolan Flamand', 'F'). Handles 'W/C', 'RW/LW', etc."""
#     if not isinstance(player_raw, str):
#         return None, None
#     # pull trailing parens content
#     m = re.search(r"\(([^)]+)\)\s*$", player_raw)
#     pos_text = m.group(1).strip().upper() if m else None
#     name = re.sub(r"\s*\([^)]+\)\s*$", "", player_raw).strip()
#     return name, pos_text

# def normalize_positions(position_text: str):
#     """Return Pos_OVR, Pos_Prime, Pos_Second, mapping 'W' → 'F'."""
#     if not isinstance(position_text, str) or not position_text.strip():
#         return None, None, None
#     tokens = [t.strip().upper() for t in position_text.split("/") if t.strip()]
#     tokens = [t for t in tokens if t in ALLOWED_POS]
#     if not tokens:
#         return None, None, None
#     # Map W → F for our schema
#     tokens = ["F" if t == "W" else t for t in tokens]
#     pos_prime = tokens[0]
#     pos_second = tokens[1] if len(tokens) > 1 else None
#     if pos_prime == "G":
#         pos_ovr = "G"
#     elif pos_prime == "D":
#         pos_ovr = "D"
#     else:
#         pos_ovr = "F"
#     return pos_ovr, pos_prime, pos_second

# def name_key(s: str) -> str | None:
#     if not isinstance(s, str) or not s.strip():
#         return None
#     s2 = unidecode(s) if unidecode else s
#     s2 = re.sub(r"[^a-zA-Z0-9]+", "", s2).lower()
#     return s2

# def parse_header_team_season(soup: BeautifulSoup):
#     """Try to extract Team and Season from the headline text."""
#     h2 = soup.find("h2", class_=re.compile("TitleWithExpandButton_titleWithExpandButton"))
#     team = None; season = None
#     if h2:
#         t = text(h2)
#         # Example: "2024-2025 Brandon Wheat Kings Player Stats"
#         m = re.search(r"(\d{4}-\d{4})", t)
#         season = m.group(1) if m else None
#         team = re.sub(r"\d{4}-\d{4}\s*", "", t)
#         team = team.replace("Player Stats", "").strip(" -")
#     return team, season

# def parse_league_from_section(table: BeautifulSoup):
#     """Look for the 'section' row that has an <a> with league text like 'WHL'."""
#     a = table.find("a", href=re.compile(r"/league/"))
#     return text(a) if a else None

# # -----------------------------
# # Core parsing: locate two tables and extract rows
# # -----------------------------
# def parse_stats_tables(html: str):
#     soup = BeautifulSoup(html, "html.parser")
#     team_hdr, season_hdr = parse_header_team_season(soup)

#     tables = soup.find_all("table")
#     skaters_tbl = None
#     goalies_tbl = None

#     for tbl in tables:
#         thead = tbl.find("thead")
#         if not thead:
#             continue
#         ths = [text(th) for th in thead.find_all("th")]
#         head_str = " ".join(ths).lower()
#         if "skater" in head_str:
#             skaters_tbl = tbl
#         elif "goalie" in head_str:
#             goalies_tbl = tbl

#     if not skaters_tbl and not goalies_tbl:
#         raise ValueError("Could not locate Skater/Goalie tables in HTML.")

#     league = parse_league_from_section(skaters_tbl or goalies_tbl)

#     def parse_table(tbl, entity_label):
#         rows = []
#         for tr in tbl.find_all("tr"):
#             # Skip section headers or empty rows
#             if "tsection" in " ".join(tr.get("class", [])):
#                 continue
#             tds = tr.find_all("td")
#             if not tds:
#                 continue
#             rows.append(tds)
#         # infer columns from thead
#         ths = [text(th) for th in tbl.find("thead").find_all("th")]
#         return ths, rows

#     sk_head, sk_rows = ([], [])
#     if skaters_tbl:
#         sk_head, sk_rows = parse_table(skaters_tbl, "Skater")
#     go_head, go_rows = ([], [])
#     if goalies_tbl:
#         go_head, go_rows = parse_table(goalies_tbl, "Goalie")

#     meta = {"Team": team_hdr, "Season": season_hdr, "League": league}
#     return (sk_head, sk_rows, skaters_tbl is not None), (go_head, go_rows, goalies_tbl is not None), meta

# # -----------------------------
# # Cleaning / normalization
# # -----------------------------
# def clean_stats_skaters(head_cells, rows, meta):
#     # Map header labels to canonical names
#     header_map = {
#         "#": "Rank",
#         "N": "Nat",
#         "Skater": "Player",
#         "GP": "GP",
#         "G": "G",
#         "A": "A",
#         "TP": "TP",
#         "PIM": "PIM",
#         "+/-": "PlusMinus",
#     }
#     # Playoff columns repeat; we will suffix _PO for the second block
#     def normalize_headers(head_cells):
#         norm = []
#         seen_gp_block = 0
#         for h in head_cells:
#             h = h.strip()
#             base = header_map.get(h, h)
#             if base in {"GP","G","A","TP","PIM","PlusMinus"}:
#                 if seen_gp_block == 0:
#                     norm.append(base)
#                 else:
#                     norm.append(base + "_PO")
#                 if base == "PlusMinus":
#                     # the block ended
#                     seen_gp_block += 1
#             else:
#                 norm.append(header_map.get(h, h))
#         return norm

#     cols = normalize_headers(head_cells) if head_cells else []
#     # Build records
#     records = []
#     for tds in rows:
#         vals = [text(td) for td in tds]
#         # Align length (in case of hidden mobile columns etc.)
#         if len(vals) != len(cols):
#             # Best effort: pad/truncate
#             if len(vals) < len(cols):
#                 vals += [""] * (len(cols) - len(vals))
#             else:
#                 vals = vals[:len(cols)]
#         rec = dict(zip(cols, vals))
#         # Extract player link & raw name
#         player_url, player_id, player_raw = extract_player_link(tds[2]) if len(tds) >= 3 else (None,None,rec.get("Player"))
#         rec["Player"] = player_raw or rec.get("Player")
#         rec["EP_Player_URL"] = player_url
#         rec["EP_Player_ID"]  = player_id
#         # Parse name + positions
#         name, pos_text = parse_player_field(rec.get("Player"))
#         rec["Player_Name"] = name
#         rec["Position_text"] = pos_text
#         pos_ovr, pos_prime, pos_second = normalize_positions(pos_text) if pos_text else (None,None,None)
#         rec["Pos_OVR"] = pos_ovr; rec["Pos_Prime"] = pos_prime; rec["Pos_Second"] = pos_second
#         # Names split + key
#         first, last = split_name(name)
#         rec["First_Name"] = first; rec["Last_Name"] = last
#         rec["Name_Key"] = name_key(name)
#         # Attach meta
#         rec |= meta
#         records.append(rec)
#     df = pd.DataFrame.from_records(records)
#     # Type fixups
#     for c in ["GP","G","A","TP","PIM","PlusMinus","GP_PO","G_PO","A_PO","TP_PO","PIM_PO","PlusMinus_PO","Rank"]:
#         if c in df.columns:
#             df[c] = pd.to_numeric(df[c].str.replace(",","", regex=False), errors="coerce")
#     return df

# def mmss_to_seconds(s: str):
#     if not isinstance(s, str) or ":" not in s:
#         return None
#     try:
#         parts = s.split(":")
#         if len(parts) == 2:
#             m, s2 = int(parts[0]), int(parts[1])
#             return m*60 + s2
#         elif len(parts) == 3:
#             h, m, s2 = int(parts[0]), int(parts[1]), int(parts[2])
#             return h*3600 + m*60 + s2
#     except Exception:
#         return None
#     return None

# def clean_stats_goalies(head_cells, rows, meta):
#     header_map = {
#         "#": "Rank",
#         "N": "Nat",
#         "Goalie": "Player",
#         "GP": "GP",
#         "GAA": "GAA",
#         "SV%": "SV%",
#         "W": "W", "L": "L", "T": "T",
#         "SO": "SO",
#         "TOI": "TOI",
#         "SVS": "SVS"
#     }
#     # Second block is playoffs: GP, GAA, SV%
#     def normalize_headers(head_cells):
#         norm = []
#         seen_gp_block = 0
#         for h in head_cells:
#             h = h.strip()
#             base = header_map.get(h, h)
#             if base in {"GP","GAA","SV%"}:
#                 if seen_gp_block == 0:
#                     norm.append(base)
#                 else:
#                     norm.append(base + "_PO")
#                 if base == "SV%":
#                     seen_gp_block += 1
#             else:
#                 norm.append(header_map.get(h, h))
#         return norm

#     cols = normalize_headers(head_cells) if head_cells else []
#     records = []
#     for tds in rows:
#         vals = [text(td) for td in tds]
#         if len(vals) != len(cols):
#             if len(vals) < len(cols):
#                 vals += [""] * (len(cols) - len(vals))
#             else:
#                 vals = vals[:len(cols)]
#         rec = dict(zip(cols, vals))
#         player_url, player_id, player_raw = extract_player_link(tds[2]) if len(tds) >= 3 else (None,None,rec.get("Player"))
#         rec["Player"] = player_raw or rec.get("Player")
#         rec["EP_Player_URL"] = player_url
#         rec["EP_Player_ID"]  = player_id
#         # Parse name + positions
#         name, pos_text = parse_player_field(rec.get("Player"))
#         rec["Player_Name"] = name
#         rec["Position_text"] = pos_text
#         pos_ovr, pos_prime, pos_second = normalize_positions(pos_text) if pos_text else (None,None,None)
#         rec["Pos_OVR"] = pos_ovr; rec["Pos_Prime"] = pos_prime; rec["Pos_Second"] = pos_second
#         # Names split + key
#         first, last = split_name(name)
#         rec["First_Name"] = first; rec["Last_Name"] = last
#         rec["Name_Key"] = name_key(name)
#         # TOI to seconds + minutes
#         if "TOI" in rec and rec["TOI"]:
#             secs = mmss_to_seconds(rec["TOI"])
#             rec["TOI_Seconds"] = secs
#             rec["TOI_Minutes"] = (secs/60.0) if secs is not None else None
#         # Save % → float
#         for k in ["SV%","SV%_PO"]:
#             if k in rec and isinstance(rec[k], str) and rec[k].startswith("."):
#                 try:
#                     rec[k] = float(rec[k])
#                 except Exception:
#                     pass
#         # Attach meta
#         rec |= meta
#         records.append(rec)
#     df = pd.DataFrame.from_records(records)
#     # numeric coercions
#     num_cols = ["GP","W","L","T","SO","SVS","GP_PO"]
#     for c in num_cols:
#         if c in df.columns:
#             df[c] = pd.to_numeric(df[c].astype(str).str.replace(",","", regex=False), errors="coerce")
#     for c in ["GAA","GAA_PO"]:
#         if c in df.columns:
#             df[c] = pd.to_numeric(df[c], errors="coerce")
#     return df

# # -----------------------------
# # Public entrypoint
# # -----------------------------
# def scrape_team_stats(team_name: str, team_url: str, season: str | None = None, refresh: bool = False):
#     # url = fix_url_stats(team_url, season=season)
#     url = 'https://www.eliteprospects.com/team/859/medicine-hat-tigers/2024-2025?tab=stats'
#     cache_name = re.sub(r'[^A-Za-z0-9]+','_', f"{team_name}_{season or 'current'}").strip("_")
#     html = fetch_html(url, cache_name=cache_name, refresh=refresh)
#     sk, go, meta = parse_stats_tables(html)
#     # prefer season from header if available
#     if meta.get("Season") is None and season:
#         meta["Season"] = season
#     if meta.get("Team") is None:
#         meta["Team"] = team_name

#     skaters_df = pd.DataFrame()
#     goalies_df = pd.DataFrame()
#     if sk[2]:
#         skaters_df = clean_stats_skaters(sk[0], sk[1], meta)
#     if go[2]:
#         goalies_df = clean_stats_goalies(go[0], go[1], meta)

#     # Write outputs
#     safe_team = re.sub(r'[^A-Za-z0-9]+','_', meta.get("Team") or team_name).strip("_")
#     season_tag = (meta.get("Season") or season or "current").replace("/","-")
#     if not skaters_df.empty:
#         sk_out = OUT_DIR / f"{safe_team}_{season_tag}_skaters.csv"
#         skaters_df.to_csv(sk_out, index=False)
#         logger.info(f"Wrote {sk_out} ({len(skaters_df)} rows)")
#     if not goalies_df.empty:
#         go_out = OUT_DIR / f"{safe_team}_{season_tag}_goalies.csv"
#         goalies_df.to_csv(go_out, index=False)
#         logger.info(f"Wrote {go_out} ({len(goalies_df)} rows)")

#     return skaters_df, goalies_df


In [ ]:
# TEAM = "Medicine Hat Tigers"
# URL  = "https://www.eliteprospects.com/team/859/medicine-hat-tigers/2024-2025?tab=stats"
# SEASON = "2024-2025"
# sk_df, go_df = scrape_team_stats(TEAM, URL, season=SEASON, refresh=False)
# display(sk_df.head())
# display(go_df.head())


In [ ]:
# Write an **async** Playwright version to avoid "using Sync API inside the asyncio loop".
# This module mirrors the earlier parser/cleaners but renders with the async API.
# You can: `from ep_team_stats_playwright_async_v1 import scrape_team_stats_rendered_async`
# and then (in Jupyter) call:
#   sk_df, go_df = await scrape_team_stats_rendered_async(TEAM, URL, season="2024-2025")
# Or in a normal .py script:
#   import asyncio
#   sk_df, go_df = asyncio.run(scrape_team_stats_rendered_async(TEAM, URL, season="2024-2025"))
from pathlib import Path


import os, re, time, random, logging
from io import StringIO
from pathlib import Path
import pandas as pd
from bs4 import BeautifulSoup

try:
    from unidecode import unidecode
except Exception:
    unidecode = None

# ---- Playwright (async) ----
from playwright.async_api import async_playwright, TimeoutError as PWTimeoutError

BASE = Path(".")
RAW_HTML_DIR = BASE / "raw_html_stats"
OUT_DIR      = BASE / "out"
LOG_DIR      = BASE / "logs"
for d in [RAW_HTML_DIR, OUT_DIR, LOG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

logger = logging.getLogger("ep_stats_async")
logger.setLevel(logging.INFO)
if not logger.handlers:
    fh = logging.FileHandler(LOG_DIR / "ep_stats_rendered_async.log")
    fh.setFormatter(logging.Formatter("%(asctime)s %(levelname)s %(message)s"))
    logger.addHandler(fh)
    ch = logging.StreamHandler()
    ch.setFormatter(logging.Formatter("%(levelname)s %(message)s"))
    logger.addHandler(ch)

ALLOWED_POS = {"G","D","F","LW","C","RW","W"}

def fix_url_stats(url: str, season: str | None = None) -> str:
    from urllib.parse import urlparse, parse_qs, urlencode, urlunparse
    u = urlparse(url)
    q = parse_qs(u.query)
    q["tab"] = ["stats"]
    if season:
        q["season"] = [season]
    new_q = urlencode({k: v[0] if isinstance(v, list) else v for k, v in q.items()})
    return urlunparse((u.scheme, u.netloc, u.path, u.params, new_q, u.fragment))

async def render_html_async(url: str, cache_name: str, refresh: bool = False, click_expand: bool = True, timeout_ms: int = 20000) -> str:
    """
    Use headless Chromium (async Playwright) to render the React page and return full HTML.
    Cache the result to avoid hammering the site.
    """
    cache_path = RAW_HTML_DIR / f"{cache_name}.rendered.html"
    if cache_path.exists() and not refresh:
        return cache_path.read_text(encoding="utf-8", errors="ignore")

    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        ctx = await browser.new_context(
            user_agent="Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0 Safari/537.36 NCAADataSauce/1.0",
            locale="en-US",
            viewport={"width": 1366, "height": 900},
        )
        page = await ctx.new_page()

        logger.info(f"Navigating to {url}")
        await page.goto(url, wait_until="domcontentloaded")

        # Ensure we're on the stats tab (defensive)
        try:
            stats_tab = page.locator('a:has-text("Stats")')
            if await stats_tab.count() > 0:
                await stats_tab.first.click()
        except Exception:
            pass

        # Some pages collapse the block; expand if button exists
        if click_expand:
            try:
                btn = page.locator('button[aria-label*="expand" i]')
                if await btn.count() > 0 and await btn.first.is_visible():
                    await btn.first.click()
            except Exception:
                pass

        # Wait for at least one stats table with rows (skaters or goalies)
        try:
            await page.wait_for_selector("table.SortTable_table__jnnJk thead", state="visible", timeout=timeout_ms)
            await page.wait_for_timeout(500)  # short pause
            await page.wait_for_selector("table.SortTable_table__jnnJk tbody tr", state="attached", timeout=timeout_ms)
        except PWTimeoutError:
            logger.warning("Timed out waiting for tables/rows; capturing whatever is present.")

        html = await page.content()
        await browser.close()

    cache_path.write_text(html, encoding="utf-8")
    return html

# -----------------------------
# Helpers for parsing/cleaning
# -----------------------------
def text(el):
    return el.get_text(strip=True) if el else ""

def extract_player_link(td):
    a = td.find("a")
    if not a:
        return None, None, text(td)
    href = a.get("href", "")
    player_url = "https://www.eliteprospects.com" + href if href.startswith("/") else href
    m = re.search(r"/player/(\d+)/", href)
    player_id = m.group(1) if m else None
    return player_url, player_id, text(td)

def split_name(full_name: str):
    if not isinstance(full_name, str) or not full_name.strip():
        return None, None
    parts = full_name.strip().split()
    if len(parts) == 1:
        return parts[0], None
    return " ".join(parts[:-1]), parts[-1]

def parse_player_field(player_raw: str):
    if not isinstance(player_raw, str):
        return None, None
    m = re.search(r"\(([^)]+)\)\s*$", player_raw)
    pos_text = m.group(1).strip().upper() if m else None
    name = re.sub(r"\s*\([^)]+\)\s*$", "", player_raw).strip()
    return name, pos_text

def normalize_positions(position_text: str):
    if not isinstance(position_text, str) or not position_text.strip():
        return None, None, None
    tokens = [t.strip().upper() for t in position_text.split("/") if t.strip()]
    tokens = [t for t in tokens if t in ALLOWED_POS]
    if not tokens:
        return None, None, None
    tokens = ["F" if t == "W" else t for t in tokens]
    pos_prime = tokens[0]
    pos_second = tokens[1] if len(tokens) > 1 else None
    if pos_prime == "G":
        pos_ovr = "G"
    elif pos_prime == "D":
        pos_ovr = "D"
    else:
        pos_ovr = "F"
    return pos_ovr, pos_prime, pos_second

def name_key(s: str) -> str | None:
    if not isinstance(s, str) or not s.strip():
        return None
    s2 = unidecode(s) if unidecode else s
    s2 = re.sub(r"[^a-zA-Z0-9]+", "", s2).lower()
    return s2

def parse_header_team_season(soup: BeautifulSoup):
    h2 = soup.find("h2")
    team = None; season = None
    if h2:
        t = text(h2)
        m = re.search(r"(\d{4}-\d{4})", t)
        season = m.group(1) if m else None
        team = re.sub(r"\d{4}-\d{4}\s*", "", t)
        team = team.replace("Player Stats", "").strip(" -")
    return team, season

def parse_league_from_section(table: BeautifulSoup):
    a = table.find("a", href=re.compile(r"/league/"))
    return text(a) if a else None

def parse_stats_tables(html: str):
    soup = BeautifulSoup(html, "html.parser")
    team_hdr, season_hdr = parse_header_team_season(soup)

    tables = soup.find_all("table")
    skaters_tbl = None
    goalies_tbl = None

    for tbl in tables:
        thead = tbl.find("thead")
        if not thead:
            continue
        ths = [text(th) for th in thead.find_all("th")]
        head_str = " ".join(ths).lower()
        if "skater" in head_str:
            skaters_tbl = tbl
        elif "goalie" in head_str:
            goalies_tbl = tbl

    if not skaters_tbl and not goalies_tbl:
        raise ValueError("Could not locate Skater/Goalie tables in HTML.")

    league = parse_league_from_section(skaters_tbl or goalies_tbl)

    def parse_table(tbl):
        rows = []
        for tr in tbl.find_all("tr"):
            if "tsection" in " ".join(tr.get("class", [])):
                continue
            tds = tr.find_all("td")
            if not tds:
                continue
            rows.append(tds)
        ths = [text(th) for th in tbl.find("thead").find_all("th")]
        return ths, rows

    sk_head, sk_rows = ([], [])
    if skaters_tbl:
        sk_head, sk_rows = parse_table(skaters_tbl)
    go_head, go_rows = ([], [])
    if goalies_tbl:
        go_head, go_rows = parse_table(goalies_tbl)

    meta = {"Team": team_hdr, "Season": season_hdr, "League": league}
    return (sk_head, sk_rows, skaters_tbl is not None), (go_head, go_rows, goalies_tbl is not None), meta

def clean_stats_skaters(head_cells, rows, meta):
    header_map = {"#":"Rank","N":"Nat","Skater":"Player","GP":"GP","G":"G","A":"A","TP":"TP","PIM":"PIM","+/-":"PlusMinus"}
    def normalize_headers(head_cells):
        norm = []
        seen_gp_block = 0
        for h in head_cells:
            h = h.strip()
            base = header_map.get(h, h)
            if base in {"GP","G","A","TP","PIM","PlusMinus"}:
                if seen_gp_block == 0:
                    norm.append(base)
                else:
                    norm.append(base + "_PO")
                if base == "PlusMinus":
                    seen_gp_block += 1
            else:
                norm.append(header_map.get(h, h))
        return norm

    cols = normalize_headers(head_cells) if head_cells else []
    records = []
    for tds in rows:
        vals = [td.get_text(strip=True) for td in tds]
        if len(vals) != len(cols):
            if len(vals) < len(cols):
                vals += [""] * (len(cols)-len(vals))
            else:
                vals = vals[:len(cols)]
        rec = dict(zip(cols, vals))
        # player link
        player_url, player_id, player_raw = extract_player_link(tds[2]) if len(tds) >= 3 else (None,None,rec.get("Player"))
        rec["Player"] = player_raw or rec.get("Player")
        rec["EP_Player_URL"] = player_url
        rec["EP_Player_ID"]  = player_id
        name, pos_text = parse_player_field(rec.get("Player"))
        rec["Player_Name"] = name
        rec["Position_text"] = pos_text
        pos_ovr, pos_prime, pos_second = normalize_positions(pos_text) if pos_text else (None,None,None)
        rec["Pos_OVR"] = pos_ovr; rec["Pos_Prime"] = pos_prime; rec["Pos_Second"] = pos_second
        first, last = split_name(name)
        rec["First_Name"] = first; rec["Last_Name"] = last
        rec["Name_Key"] = name_key(name)
        rec |= meta
        records.append(rec)
    df = pd.DataFrame.from_records(records)
    for c in ["GP","G","A","TP","PIM","PlusMinus","GP_PO","G_PO","A_PO","TP_PO","PIM_PO","PlusMinus_PO","Rank"]:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c].astype(str).str.replace(",","", regex=False), errors="coerce")
    return df

def mmss_to_seconds(s: str):
    if not isinstance(s, str) or ":" not in s:
        return None
    try:
        parts = s.split(":")
        if len(parts) == 2:
            m, s2 = int(parts[0]), int(parts[1])
            return m*60 + s2
        elif len(parts) == 3:
            h, m, s2 = int(parts[0]), int(parts[1]), int(parts[2])
            return h*3600 + m*60 + s2
    except Exception:
        return None
    return None

def clean_stats_goalies(head_cells, rows, meta):
    header_map = {"#":"Rank","N":"Nat","Goalie":"Player","GP":"GP","GAA":"GAA","SV%":"SV%","W":"W","L":"L","T":"T","SO":"SO","TOI":"TOI","SVS":"SVS"}
    def normalize_headers(head_cells):
        norm = []
        seen_gp_block = 0
        for h in head_cells:
            h = h.strip()
            base = header_map.get(h, h)
            if base in {"GP","GAA","SV%"}:
                if seen_gp_block == 0:
                    norm.append(base)
                else:
                    norm.append(base + "_PO")
                if base == "SV%":
                    seen_gp_block += 1
            else:
                norm.append(header_map.get(h, h))
        return norm

    cols = normalize_headers(head_cells) if head_cells else []
    records = []
    for tds in rows:
        vals = [td.get_text(strip=True) for td in tds]
        if len(vals) != len(cols):
            if len(vals) < len(cols):
                vals += [""] * (len(cols)-len(vals))
            else:
                vals = vals[:len(cols)]
        rec = dict(zip(cols, vals))
        player_url, player_id, player_raw = extract_player_link(tds[2]) if len(tds) >= 3 else (None,None,rec.get("Player"))
        rec["Player"] = player_raw or rec.get("Player")
        rec["EP_Player_URL"] = player_url
        rec["EP_Player_ID"]  = player_id
        name, pos_text = parse_player_field(rec.get("Player"))
        rec["Player_Name"] = name
        rec["Position_text"] = pos_text
        pos_ovr, pos_prime, pos_second = normalize_positions(pos_text) if pos_text else (None,None,None)
        rec["Pos_OVR"] = pos_ovr; rec["Pos_Prime"] = pos_prime; rec["Pos_Second"] = pos_second
        first, last = split_name(name)
        rec["First_Name"] = first; rec["Last_Name"] = last
        rec["Name_Key"] = name_key(name)
        if "TOI" in rec and rec["TOI"]:
            secs = mmss_to_seconds(rec["TOI"])
            rec["TOI_Seconds"] = secs
            rec["TOI_Minutes"] = (secs/60.0) if secs is not None else None
        for k in ["SV%","SV%_PO"]:
            if k in rec and isinstance(rec[k], str) and rec[k].startswith("."):
                try:
                    rec[k] = float(rec[k])
                except Exception:
                    pass
        rec |= meta
        records.append(rec)
    df = pd.DataFrame.from_records(records)
    for c in ["GP","W","L","T","SO","SVS","GP_PO"]:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c].astype(str).str.replace(",","", regex=False), errors="coerce")
    for c in ["GAA","GAA_PO"]:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")
    return df

async def scrape_team_stats_rendered_async(team_name: str, team_url: str, season: str | None = None, refresh: bool = False):
    """
    Fully rendered scrape (Async Playwright) + parse + clean. Returns (skaters_df, goalies_df).
    Use `await` in notebooks, or `asyncio.run(...)` in scripts.
    """
    from urllib.parse import urlparse
    url = fix_url_stats(team_url, season=season)
    parsed = urlparse(url)
    id_part = "_".join([p for p in parsed.path.split("/") if p.isdigit()]) or "team"
    cache_name = re.sub(r'[^A-Za-z0-9]+','_', f"{id_part}_{season or 'current'}").strip("_")

    html = await render_html_async(url, cache_name=cache_name, refresh=refresh)
    sk, go, meta = parse_stats_tables(html)
    if meta.get("Season") is None and season:
        meta["Season"] = season
    if meta.get("Team") is None:
        meta["Team"] = team_name

    skaters_df = pd.DataFrame()
    goalies_df = pd.DataFrame()
    if sk[2] and len(sk[1]) > 0:
        skaters_df = clean_stats_skaters(sk[0], sk[1], meta)
    if go[2] and len(go[1]) > 0:
        goalies_df = clean_stats_goalies(go[0], go[1], meta)

    safe_team = re.sub(r'[^A-Za-z0-9]+','_', meta.get("Team") or team_name).strip("_")
    season_tag = (meta.get("Season") or season or "current").replace("/","-")
    if not skaters_df.empty:
        sk_out = OUT_DIR / f"{safe_team}_{season_tag}_skaters.csv"
        skaters_df.to_csv(sk_out, index=False)
        logger.info(f"Wrote {sk_out} ({len(skaters_df)} rows)")
    if not goalies_df.empty:
        go_out = OUT_DIR / f"{safe_team}_{season_tag}_goalies.csv"
        goalies_df.to_csv(go_out, index=False)
        logger.info(f"Wrote {go_out} ({len(goalies_df)} rows)")

    return skaters_df, goalies_df


In [ ]:


TEAM   = "Medicine Hat Tigers"
URL    = "https://www.eliteprospects.com/team/859/medicine-hat-tigers/2024-2025?tab=stats"
SEASON = "2024-2025"

# Jupyter/IPython supports top-level `await`
sk_df, go_df = await scrape_team_stats_rendered_async(TEAM, URL, season=SEASON, refresh=False)
display(sk_df.head())
display(go_df.head())




In [ ]:
dfs[15].head(10)

In [ ]:
## Save as a temp csv file for data cleaning look


# dfs[0].to_csv(output_local, index=False)
